# ExplaGraphs on Kaggle: full graphs, preprocessing through test evaluation

Run top to bottom with **Internet** and **GPU T4 x2**. This experiment restores the repository's original ExplaGraphs graph path: parse every explanation triple, embed all nodes/relations, and give the model both the **complete graph** and the **complete node/edge CSV description**. There is **no 2-hop extraction, retrieval, ranking, top-k selection, pruning or ego-graph merging**.

The source TSV and original 60/20/20 split procedure (`random_state=42`) are preserved. Sentence embeddings use `all-roberta-large-v1`, FP32 mean pooling and normalization. Queries correctly use each row's arguments, matching the original dataset loader rather than the preprocessing script's accidental full-column question formatting.

**Scope:** this reproduces the original graph representation, not necessarily the paper's exact training experiment. As in the recent notebooks, Llama-2-7b-chat uses NF4 QLoRA plus the original GAT/projector. The author's CLI defaults instead freeze the unquantized LLM. Quantization, chat checkpoint, native libraries and full-text handling can affect scores; a 92.74% result is not guaranteed. Compare this run against your retrieval run without selecting settings using test accuracy.

Full training uses **10 epochs**, **patience 5**, batch 2 and accumulation 2 (effective batch 4), matching the repository's epoch/early-stop/batch defaults. Smoke mode defaults on: it preprocesses and backs up the **whole dataset**, but trains/evaluates only tiny subsets for one epoch. Turn it off for reportable held-out results.

Drive conventions are unchanged below the selected roots: `GRAG preprocessing backups/expla_graphs-<snapshot-id>/` and `GRAG downstream/expla_graphs/<full-graph-run-name>/`. The roots may live in My Drive, another account's shared folder, or a Google Shared Drive. A separate full-graph run name prevents overwriting retrieval checkpoints.

## 1. Runtime and repository

Push/upload this notebook and its accompanying `expla_full_graph.py` helper before cloning remotely. Keep Kaggle's Python/PyTorch/CUDA; do not recreate the author's environment. For resume, use the same code commit and settings. This notebook does not modify the old retrieval artifacts or your running WebQSP experiment.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

os.environ['HF_HOME'] = '/kaggle/temp/huggingface'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Select Kaggle GPU T4 x2.')
for i in range(2):
    props = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {props.name}, {props.total_memory / 2**30:.2f} GiB')
ROOT = Path('/kaggle/working/BioGRAG-QA')
REPO_URL = 'https://github.com/MichaelPham2005/BioGRAG-QA.git'
REPO_BRANCH = 'GRAG/reproduce/WebQSP'
CODE_REVISION = ''  # Set the original Git commit for resume after branch updates.
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(ROOT)], check=True)
if CODE_REVISION:
    actual = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    requested = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', CODE_REVISION], text=True).strip()
    if actual != requested:
        subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', CODE_REVISION], check=True)
if not (ROOT / 'src/utils/notebook_preprocessing/expla_full_graph.py').is_file():
    raise FileNotFoundError('Updated full-graph helpers are missing; push/upload them first.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COMMIT = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Code:', COMMIT, 'PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)

## 2. Install only missing packages

If Kaggle requests a Python restart, restart and rerun from section 1. No sentence-transformers wrapper, legacy gensim import, torch-scatter compilation or old CUDA installation is needed.

In [ ]:
requirements = {
    'torch_geometric': 'torch-geometric', 'transformers': 'transformers',
    'accelerate': 'accelerate', 'bitsandbytes': 'bitsandbytes', 'peft': 'peft',
    'sentencepiece': 'sentencepiece', 'pandas': 'pandas', 'sklearn': 'scikit-learn',
    'tqdm': 'tqdm', 'requests': 'requests', 'huggingface_hub': 'huggingface-hub',
    'googleapiclient': 'google-api-python-client', 'google.auth': 'google-auth',
}
missing = []
for module, package in requirements.items():
    try:
        present = importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        present = False
    if not present:
        missing.append(package)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    importlib.invalidate_caches()
from importlib.metadata import version
from src.utils.notebook_preprocessing import kaggle_preprocess as preprocessing
from src.utils.notebook_preprocessing import expla_full_graph as full_graph
from src.utils.notebook_preprocessing.drive_backup import backup_to_drive, ensure_folder, find_remote, resolve_folder
from src.utils.notebook_downstream import artifacts, training
from src.utils.notebook_downstream.drive_sync import DriveFolder, restore_drive_snapshot
print({name: version(name) for name in requirements.values()})

## 3. Secrets, experiment name and existing Drive folders

Accept the gated Llama-2 license and enable Kaggle Secrets **HF_TOKEN**, **GDRIVE_CLIENT_ID**, **GDRIVE_CLIENT_SECRET**, **GDRIVE_REFRESH_TOKEN**. Use the newly authorized **full-Drive** token from the same desktop OAuth client, not the old preprocessing-only `drive.file` token. Never paste credentials into cells or outputs. Kaggle has no Colab-style Drive filesystem mount.

The two root folder names are the same as Colab; missing/duplicate roots stop explicitly. Leave both IDs as `None` to discover folders or shortcuts named **GRAG preprocessing backups** and **GRAG downstream** directly under your My Drive. For another account's shared folders or a Google Shared Drive, set `PREPROCESSING_ROOT_ID` and `DOWNSTREAM_ROOT_ID` to those two actual folder IDs (shortcut IDs also work). Do not supply a common parent ID: the target folders must have those exact names and allow creating children. Each user needs their own full-Drive OAuth authorization and write access; a shortcut does not grant permissions.

A saved `preprocessing.json` links each run to its exact full-graph snapshot, avoiding confusion with older retrieval snapshots. Set a distinct `RUN_OWNER` for each collaborator before the first run; leave it unchanged for resume. Empty preserves existing run names. Do not run two sessions writing this same run folder.

In [ ]:
import re
from kaggle_secrets import UserSecretsClient
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from google.auth.exceptions import RefreshError
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from huggingface_hub import HfApi, hf_hub_download
from src.utils.notebook_preprocessing.drive_backup import ensure_folder, find_remote, resolve_folder

SMOKE_TEST = True
DATASET = 'expla_graphs'
RUN_OWNER = ''  # Optional stable collaborator tag, e.g. 'experiment-b'; never a token.
if RUN_OWNER and not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_OWNER):
    raise ValueError('RUN_OWNER must contain only letters, digits, underscores or hyphens.')
RUN_NAME = 'expla-graphs-full-graph-chat-qlora-kaggle-2t4-' + ('smoke-v1' if SMOKE_TEST else 'full-v1')
if RUN_OWNER:
    RUN_NAME += '-' + RUN_OWNER
MODEL_ID = 'meta-llama/Llama-2-7b-chat-hf'
PREPROCESSING_ROOT_ID = None
DOWNSTREAM_ROOT_ID = None
if PREPROCESSING_ROOT_ID is not None and not re.fullmatch(r'[A-Za-z0-9_-]+', PREPROCESSING_ROOT_ID):
    raise ValueError('Use a folder ID, not a Drive URL.')
if DOWNSTREAM_ROOT_ID is not None and not re.fullmatch(r'[A-Za-z0-9_-]+', DOWNSTREAM_ROOT_ID):
    raise ValueError('Use a folder ID, not a Drive URL.')
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Enable HF_TOKEN and accept the Llama-2 license.')
hf_hub_download(MODEL_ID, 'config.json', token=HF_TOKEN)
credentials = Credentials(
    token=None, token_uri='https://oauth2.googleapis.com/token',
    client_id=secrets.get_secret('GDRIVE_CLIENT_ID'),
    client_secret=secrets.get_secret('GDRIVE_CLIENT_SECRET'),
    refresh_token=secrets.get_secret('GDRIVE_REFRESH_TOKEN'),
    scopes=['https://www.googleapis.com/auth/drive'],
)
try:
    credentials.refresh(Request())
except RefreshError as error:
    code = next((item.get('error') for item in error.args if isinstance(item, dict)), None)
    if code == 'invalid_scope':
        message = 'Drive invalid_scope: use the newly authorized full-Drive refresh token, not the old drive.file token.'
    elif code == 'invalid_grant':
        message = 'Drive invalid_grant: renew expired/revoked offline consent using full Drive scope and the matching desktop client.'
    elif code in ('invalid_client', 'unauthorized_client'):
        message = 'Drive client rejected: all three Google secrets must come from the same desktop client/token pair.'
    elif code in ('temporarily_unavailable', 'server_error'):
        message = 'Google OAuth is temporarily unavailable; retry this cell later.'
    else:
        message = 'Drive refresh failed; check the matching secrets and account/app restrictions. Raw responses are withheld.'
    raise RuntimeError(message) from None
if credentials.granted_scopes is not None and 'https://www.googleapis.com/auth/drive' not in credentials.granted_scopes:
    raise RuntimeError('Full Drive scope was not granted; no folders were created.')
service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
roots = {}
for name in ('GRAG preprocessing backups', 'GRAG downstream'):
    configured_id = PREPROCESSING_ROOT_ID if name == 'GRAG preprocessing backups' else DOWNSTREAM_ROOT_ID
    try:
        info = resolve_folder(service, name, configured_id)
    except HttpError as error:
        if error.resp.status not in (403, 404):
            raise
        raise RuntimeError(f'Cannot access Drive folder {name}; verify the account, Shared Drive sharing, and full Drive scope.') from None
    roots[name] = info['id']
    location = 'Google Shared Drive' if info.get('driveId') else 'My Drive or shared folder'
    print(f'{name} ({location}): https://drive.google.com/drive/folders/{info["id"]}')
dataset_folder_id = ensure_folder(service, DATASET, roots['GRAG downstream'])
run_folder_id = ensure_folder(service, RUN_NAME, dataset_folder_id)
drive_store = DriveFolder(service, run_folder_id, Path('/kaggle/temp/drive') / roots['GRAG downstream'] / DATASET / RUN_NAME)
DRIVE_OUTPUT = drive_store.cache
drive_store.restore_training_outputs()
drive_store.download('preprocessing.json', required=False)
OUTPUT = ROOT / 'output/downstream' / DATASET / RUN_NAME
artifacts.restore_training_outputs(DRIVE_OUTPUT, OUTPUT)
saved_run = json.loads((OUTPUT / 'run.json').read_text()) if (OUTPUT / 'run.json').exists() else None
print('Run path below the selected downstream root:', f'{DATASET}/{RUN_NAME}/')
print('Local API cache (not a Drive mount):', DRIVE_OUTPUT)
print('Run folder:', f'https://drive.google.com/drive/folders/{run_folder_id}')

## 4. Prepare, embed all graphs on both T4s, verify and back up

On the first run, read the checked-in `dataset/expla_graphs/train_dev.tsv`, preserving every row and label. Two independent subprocesses embed alternating rows on separate GPUs. Node features are saved once per node; repeated relation embeddings may share storage, but **every edge occurrence and its direction are preserved**. These workers finish and release GPU memory before Llama is loaded.

The completed snapshot uses the existing bounded archive/manifest backup helper (256-MiB source-data parts, verified uploads, manifest last). A receipt `preprocessing.json` is synced to the training run. On later sessions that receipt restores the exact snapshot and skips embedding. Smoke and full training use separate receipts, but the same deterministic preprocessing can reuse an identical snapshot.

A timeout before the receipt is uploaded may require preprocessing again. Local workers resume valid sample files using the encoder revision saved in `config.json`, even if the encoder's latest revision changes. Incomplete preprocessing is never published as a complete backup. Errors stop rather than falling back to local-only persistence. Only this notebook's owned worker processes are terminated on interruption.

The notebook never rewrites helper source files. Existing local configurations produced by older/different helper code remain protected by source hashes: keep that original checkout to resume them or select a fresh preprocessing directory. Completed snapshots already linked by a receipt remain restorable.

In [ ]:
import time
receipt_path = DRIVE_OUTPUT / 'preprocessing.json'
if receipt_path.exists():
    receipt = json.loads(receipt_path.read_text())
    if receipt['pipeline'] != full_graph.PIPELINE:
        raise ValueError('This run points to a different graph pipeline; choose a fresh run name.')
    snapshot_id = receipt['snapshot_folder_id']
    PREPARED, PREPROCESS_RUN, SNAPSHOT_HASH = restore_drive_snapshot(
        service, snapshot_id, Path('/kaggle/temp/grag-snapshots') / snapshot_id,
        Path('/kaggle/temp/grag-restored') / snapshot_id, DATASET,
    )
    if SNAPSHOT_HASH != receipt['snapshot_sha256']:
        raise ValueError('The preprocessing snapshot differs from the saved receipt.')
else:
    PREPARED = ROOT / 'output/expla_graphs-full-graph-prepared'
    PREPROCESS_RUN = ROOT / 'output/expla_graphs-full-graph-fp32-v4'
    raw = ROOT / 'dataset/expla_graphs/train_dev.tsv'
    if not raw.is_file():
        raise FileNotFoundError('The repository must contain the original train_dev.tsv.')
    if not (PREPARED / 'prepared.json').exists():
        preprocessing.prepare(DATASET, PREPARED, raw=raw)
    metadata, records = preprocessing.read_prepared(PREPARED)
    if metadata['source']['sha256'] != preprocessing.digest(raw):
        raise ValueError('Raw TSV changed; use a new preprocessing directory.')
    old_config = PREPROCESS_RUN / 'config.json'
    encoder_revision = (json.loads(old_config.read_text())['model_revision'] if old_config.exists()
                        else HfApi().model_info(preprocessing.MODEL).sha)
    config = full_graph.configure_run(PREPARED, PREPROCESS_RUN, model_revision=encoder_revision)
    processes, logs = [], []
    try:
        for shard in range(config['num_shards']):
            log_path = PREPROCESS_RUN / f'shard-{shard}.log'
            logs.append(log_path)
            env = {**os.environ, 'HF_TOKEN': HF_TOKEN, 'CUDA_VISIBLE_DEVICES': str(shard), 'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2'}
            with log_path.open('a') as log:
                processes.append(subprocess.Popen(
                    [sys.executable, '-u', '-m', 'src.utils.notebook_preprocessing.expla_full_graph',
                     '--output', str(PREPROCESS_RUN), '--shard', str(shard)],
                    cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT,
                ))
        while True:
            codes = [process.poll() for process in processes]
            completed = len(list((PREPROCESS_RUN / 'samples').glob('*.pt')))
            print(time.strftime('%H:%M:%S'), f'Embedded {completed}/{len(records)} graphs', codes, flush=True)
            if any(code not in (None, 0) for code in codes):
                for path in logs:
                    with path.open('rb') as handle:
                        handle.seek(0, 2)
                        handle.seek(max(0, handle.tell() - 3000))
                        print(path.name, handle.read().decode(errors='replace'))
                raise RuntimeError('A full-graph embedding worker failed; inspect the logs above.')
            if all(code == 0 for code in codes):
                break
            time.sleep(20)
    finally:
        for process in processes:
            if process.poll() is None:
                process.terminate()
        for process in processes:
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
    dataset = artifacts.CompactTrainingDataset(PREPARED, PREPROCESS_RUN)
    full_graph.validate_dataset(dataset)
    provenance = PREPROCESS_RUN / 'session-full-graph-provenance.json'
    if not provenance.exists():
        artifacts.atomic_json({'source_commit': COMMIT, 'pipeline': full_graph.PIPELINE,
                               'source': metadata['source'], 'encoder_revision': encoder_revision,
                               'packages': {name: version(name) for name in requirements.values()}}, provenance)
    snapshot_url = backup_to_drive(PREPARED, PREPROCESS_RUN, service,
                                   root_id=roots['GRAG preprocessing backups'])
    snapshot_id = snapshot_url.rsplit('/', 1)[-1]
    manifest = find_remote(service, 'manifest.json', snapshot_id)
    if manifest is None:
        raise FileNotFoundError('The preprocessing completion manifest was not published.')
    manifest_path = OUTPUT / 'snapshot-manifest.json'
    manifest_path.write_bytes(service.files().get_media(fileId=manifest['id'], supportsAllDrives=True).execute())
    SNAPSHOT_HASH = artifacts.checksum(manifest_path)
    receipt = {'pipeline': full_graph.PIPELINE, 'snapshot_folder_id': snapshot_id,
               'snapshot_sha256': SNAPSHOT_HASH, 'source_sha256': metadata['source']['sha256']}
training.save_and_backup(receipt, 'preprocessing.json', OUTPUT, DRIVE_OUTPUT, backup=drive_store.backup_file)
dataset = artifacts.CompactTrainingDataset(PREPARED, PREPROCESS_RUN)
full_graph.validate_dataset(dataset)
original_splits = dataset.get_idx_split()
limits = {'train': 8, 'val': 4, 'test': 4}
splits = {name: list(ids[:limits[name]] if SMOKE_TEST else ids) for name, ids in original_splits.items()}
print('All source splits:', {name: len(ids) for name, ids in original_splits.items()})
print('Selected training/evaluation splits:', {name: len(ids) for name, ids in splits.items()})
print('First complete CSV graph:\n', dataset.read_payload(0)['desc'])

## 5. Pin Llama and verify that the whole graph fits, without truncation

The author's CLI caps descriptions at 512 tokens and generation at 32 tokens. Here the **full description is required**, so the description/total ceiling is the pinned model's native context (4,096 for Llama-2), with 32 answer tokens reserved. Inputs are padded only to their actual batch length, not to 4,096. Before loading the 7B model, every graph description is checked against its actual remaining question/delimiter/answer budget. If any graph cannot fit, the notebook stops; it does not silently drop nodes or edges from context.

Llama decoder layers are split over the two GPUs in a single process; embeddings/GAT/projector are on GPU 0 and final norm/head on GPU 1. This is model parallelism, not DDP or pooled 32-GB memory. It gives memory headroom, not a speed/fit guarantee. QLoRA trains rank-8 q/v adapters; GAT/projector also train. Ten epochs is an upper bound because validation-loss early stopping remains enabled.

In [ ]:
import random
from types import SimpleNamespace
from transformers import AutoConfig, AutoTokenizer
requested_revision = saved_run['settings']['model_revision'] if saved_run else 'main'
model_revision = HfApi(token=HF_TOKEN).model_info(MODEL_ID, revision=requested_revision).sha
config = AutoConfig.from_pretrained(MODEL_ID, revision=model_revision, token=HF_TOKEN)
if config.model_type != 'llama' or config.tie_word_embeddings or config.num_hidden_layers < 2:
    raise ValueError('This device map requires Llama with untied embeddings and at least two decoder layers.')
settings = training.Settings(
    dataset=DATASET, model_id=MODEL_ID, model_revision=model_revision,
    epochs=1 if SMOKE_TEST else 10, patience=5, batch_size=2, grad_accum=2,
    max_sequence_tokens=config.max_position_embeddings,
    max_description_tokens=config.max_position_embeddings, max_answer_tokens=32,
    checkpoint_steps=50,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_revision, token=HF_TOKEN, use_fast=False)
preflight = SimpleNamespace(settings=settings, tokenizer=tokenizer)
report = {name: training.token_report(preflight, dataset, ids) for name, ids in original_splits.items()}
print('Full-dataset token report:', report)
if any(count for split_report in report.values() for count in split_report.values()):
    raise ValueError('A full description/target would be truncated. Stop: this run requires every graph to fit intact.')
SIGNATURE = training.initialize_run(settings, dataset, splits, SNAPSHOT_HASH, OUTPUT, DRIVE_OUTPUT, backup=drive_store.backup_file)
training.save_and_backup(report, 'token-report.json', OUTPUT, DRIVE_OUTPUT, backup=drive_store.backup_file)
random.seed(settings.seed)
torch.manual_seed(settings.seed)
torch.cuda.manual_seed_all(settings.seed)
split_layer = config.num_hidden_layers // 2
device_map = {'model.embed_tokens': 0, 'model.norm': 1, 'lm_head': 1}
device_map.update({f'model.layers.{i}': 0 if i < split_layer else 1 for i in range(config.num_hidden_layers)})
model = training.load_qlora(settings, HF_TOKEN, device_map=device_map)
parameter_devices = {parameter.device for parameter in model.llm.parameters()}
if parameter_devices != {torch.device('cuda:0'), torch.device('cuda:1')}:
    raise RuntimeError(f'Expected both GPUs without CPU/disk offload, got {parameter_devices}.')
runtime = {'code_commit': COMMIT, 'pipeline': full_graph.PIPELINE, 'device_map': device_map,
           'gpu_names': [torch.cuda.get_device_name(i) for i in range(2)],
           'packages': {name: version(name) for name in requirements.values()}}
training.save_and_backup(runtime, 'runtime.json', OUTPUT, DRIVE_OUTPUT, backup=drive_store.backup_file)
print('Pinned model:', model_revision, 'native context:', config.max_position_embeddings)
print('Trainable/total LLM parameters:', model.llm.get_nb_trainable_parameters())

## 6. Train, validate, and sync checkpoints to Drive

Progress bars display optimizer updates, loss, learning rate, elapsed time and ETA; epoch summaries include training/validation/backup durations. `last.pt` is synced every 50 optimizer steps and at epoch boundaries; `best.pt` is selected **only by validation loss**. The checkpoints contain adapters, graph modules, optimizer/scaler, RNG for both GPUs and partial-epoch state, not the frozen Llama weights.

To resume after timeout, rerun sections 1-6 with the same run name/settings/code on two GPUs. The preprocessing receipt restores the full-graph snapshot; `last.pt` resumes training rather than restarting. Never substitute `best.pt` for `last.pt`. Drive errors are explicit and local files are retained. Up to the last unsynced interval can be lost on abrupt termination.

For the full experiment, set `SMOKE_TEST=False` in section 3 and start a fresh GPU session after the smoke check. This selects the full-v1 run folder and all original train/val/test IDs. If memory is insufficient, batch 1 with accumulation 4 retains effective batch 4, but use a fresh run name; do not change a fingerprinted run in place.

In [ ]:
for i in range(2):
    torch.cuda.reset_peak_memory_stats(i)
started = time.monotonic()
history = training.fit(model, dataset, splits, SIGNATURE, OUTPUT, DRIVE_OUTPUT, backup=drive_store.backup_file)
print(f'Training/resume elapsed: {(time.monotonic() - started) / 60:.1f} minutes')
print('Peak allocated GiB:', {i: round(torch.cuda.max_memory_allocated(i) / 2**30, 2) for i in range(2)})

## 7. Evaluate the held-out test split and sync predictions/metrics

The shared evaluator reloads `best.pt`, generates greedily (up to 32 new tokens), and applies the author's first `support`/`counter` match rule. Raw predictions are synced every ten examples and on completion. The full run evaluates **all original test IDs**; smoke results are explicitly marked as subsets. Do not tune on test accuracy.

Files retain the existing names: `run.json`, `last.pt`, `best.pt`, `history.json`, `token-report.json`, `runtime.json`, `prediction-config.json`, `predictions.json`, `metrics.json`. Test outputs live in `evaluation-best-<checkpoint-hash>/` below the run, preventing stale predictions after continued training. `preprocessing.json` adds the immutable snapshot link.

For evaluation-only recovery, run sections 1-5 and then this section; skip section 6. The saved best checkpoint and cached predictions are restored from Drive. No embedding or training is needed again.

In [ ]:
best_path = OUTPUT / 'best.pt'
if not best_path.is_file():
    raise FileNotFoundError('No best.pt found. Complete at least one train/validation epoch or restore this run.')
evaluation_name = 'evaluation-best-' + artifacts.checksum(best_path)[:16]
evaluation_id = ensure_folder(service, evaluation_name, run_folder_id)
evaluation_store = DriveFolder(service, evaluation_id, DRIVE_OUTPUT / evaluation_name)
for name in ('prediction-config.json', 'predictions.json', 'metrics.json'):
    evaluation_store.download(name, required=False)
evaluation_output = OUTPUT / evaluation_name
artifacts.restore_training_outputs(evaluation_store.cache, evaluation_output)
artifacts.backup_file(best_path, evaluation_output / 'best.pt')
artifacts.backup_file(OUTPUT / 'run.json', evaluation_output / 'run.json')
rows, metrics = training.predict_test(
    model, dataset, splits['test'], SIGNATURE, evaluation_output, evaluation_store.cache,
    backup=evaluation_store.backup_file,
)
print('SMOKE SUBSET' if SMOKE_TEST else 'FULL HELD-OUT TEST', json.dumps(metrics, indent=2))
print('Evaluation Drive folder:', f'https://drive.google.com/drive/folders/{evaluation_id}')